In [231]:
# imports
import collections
import contextlib
import sys
import wave
# !pip install webrtcvad
# import webrtcvad
import audioop
# !pip install pylangacq
import glob
import os
import math
import time

import re
import csv
import pylangacq
import numpy as np
import pandas as pd
from pathlib import Path
import contextlib
import re
import wave
# !pip install mutagen
from mutagen.mp3 import MP3

import numpy as np
np.random.seed(42)
# p = np.random.permutation(108) # n_samples = 108
# p_subjects = np.random.RandomState(seed=0).permutation(242)
# import tensorflow as tf
# from tensorflow.keras import layers
# from tensorflow.keras.models import Model
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression


# Extracting features and processing data

In [232]:
exp_features_df = pd.read_csv(r"D:\masteruwefduyqeahfdqe\ASR-project\egemaps_selected_features_of_interest.csv")

In [233]:
exp_features = pd.read_csv(r"C:\Users\DELL\Downloads\egemaps_features.csv")
exp_features_used = pd.read_csv(r"C:\Users\DELL\Downloads\egemaps_selected_feature_explanations.csv")
exp_features_explanations = pd.read_csv(r"C:\Users\DELL\Downloads\egemaps_selected_feature_explanations.csv")

In [234]:
exp_features_pitts = exp_features_df[exp_features_df['dataset']=='Pitt'].drop(columns=['dataset', 'label_name', 'wav_path', 'status', 'error'])
exp_features_gree = exp_features_df[exp_features_df['dataset']=='Greek_DemCare'].drop(columns=['dataset', 'label_name', 'wav_path', 'status', 'error'])
exp_features_chinese = exp_features_df[exp_features_df['dataset']=='Mandarin_Chou'].drop(columns=['dataset', 'label_name', 'wav_path', 'status', 'error'])

In [ ]:
def extract_sample_id(stem: str, language: str) -> str:
    """
    Convert a filename stem into a normalized sample ID that can be matched
    across datasets with different naming conventions.
    For example: '006_Daddy_patient' -> '006_Daddy' or '012-aa(01)' -> '012-aa'
    """

    stem = str(stem)

    # remove known suffix markers
    if language=="Mandarin" or language=="Greek":
        if "_patient" in stem:
            return stem.split("_patient")[0]
        # elif "_p" in stem:
        #     return stem.split("_p")[0]
        else:           
            return stem
    else:
        # first five
        return stem[:5]

# Data processing

## Interpetable features extraction

In [255]:
cha_path = Path(r"D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha")

In [256]:
language = "English"

In [257]:
from pathlib import Path
from collections import defaultdict
import pandas as pd

# root folder containing dementia/ and control/
# example: cha_path = Path("Pitt-cha")
# make sure this is defined correctly in your notebook/session
# cha_path = Path("Pitt-cha")

# roots inside each class folder

wav_root = Path("wav_patient_only")
cha_root = Path("cha")

# map class folder name -> label
# folder structure:
# Pitt-cha/dementia/cha
# Pitt-cha/dementia/wav_patient_only
# Pitt-cha/control/cha
# Pitt-cha/control/wav_patient_only
label_map = {
    "Dementia": 1,
    "Control": 0,
}

rows = []

for class_name, label in label_map.items():
    print(f"\nProcessing class '{class_name}' with label {label}...")
    # merge paths
    wav_dir = cha_path / class_name / wav_root
    cha_dir = cha_path / class_name / cha_root
    print("\nDirectories:")
    print(" wav:", wav_dir)
    print(" cha:", cha_dir)

    # collect files by full stem
    wav_files = {p.stem: p for p in wav_dir.rglob("*.wav")}
    cha_files = {p.stem: p for p in cha_dir.rglob("*.cha")}
    print(f"Collected files, wav: {len(wav_files)}, cha: {len(cha_files)}")

    # build prefix -> list[path] maps using first 5 chars
    wav_map = defaultdict(list)
    cha_map = defaultdict(list)

    for stem, path in wav_files.items():
        stem_id = extract_sample_id(stem, language=language)
        wav_map[stem_id].append(path)

    for stem, path in cha_files.items():
        stem_id = extract_sample_id(stem, language=language)
        cha_map[stem_id].append(path)

    common_ids = sorted(set(wav_map.keys()) & set(cha_map.keys()))
    print("Matched prefix ids:")
    print(len(common_ids))
    print(common_ids[:5])

    # unmatched prefixes
    missing_wav = sorted(set(cha_map.keys()) - set(wav_map.keys()))
    missing_cha = sorted(set(wav_map.keys()) - set(cha_map.keys()))

    if missing_wav:
        print(f"[WARN] {class_name}: {len(missing_wav)} .cha prefixes without matching .wav")
        print(missing_wav[:10])

    if missing_cha:
        print(f"[WARN] {class_name}: {len(missing_cha)} .wav prefixes without matching .cha")
        print(missing_cha[:10])

    # optional: report ambiguous prefixes
    ambiguous = []
    for file_id in common_ids:
        if len(wav_map[file_id]) != 1 or len(cha_map[file_id]) != 1:
            ambiguous.append(file_id)

    if ambiguous:
        print(f"[WARN] {class_name}: {len(ambiguous)} ambiguous prefix matches")
        print(ambiguous[:10])

    # only keep unambiguous 1-to-1 matches
    for file_id in common_ids:
        if len(wav_map[file_id]) == 1 and len(cha_map[file_id]) == 1:
            rows.append({
                "file_id": file_id,
                "wav_path": str(wav_map[file_id][0]),
                "cha_path": str(cha_map[file_id][0]),
                "label": label,
                "class_name": class_name,
            })
        else:
            print(f"[SKIP] Ambiguous match for prefix {file_id}")
            print("  wav:", [str(p) for p in wav_map[file_id]])
            print("  cha:", [str(p) for p in cha_map[file_id]])

files = pd.DataFrame(rows)

print("\nResult preview:")
print(files.head())
print()
print("Shape:", files.shape)
print(files["label"].value_counts(dropna=False))

# optional save
# files.to_csv("pitt_cookie_file_index.csv", index=False)


Processing class 'Dementia' with label 1...

Directories:
 wav: D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\Dementia\wav_patient_only
 cha: D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\Dementia\cha
Collected files, wav: 309, cha: 309
Matched prefix ids:
309
['001-0', '001-2', '003-0', '005-0', '005-2']

Processing class 'Control' with label 0...

Directories:
 wav: D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\Control\wav_patient_only
 cha: D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\Control\cha
Collected files, wav: 242, cha: 243
Matched prefix ids:
242
['002-0', '002-1', '002-2', '002-3', '006-2']
[WARN] Control: 1 .cha prefixes without matching .wav
['304-1']

Result preview:
  file_id                                           wav_path  \
0   001-0  D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...   
1   001-2  D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...   
2   003-0  D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...   
3   005-0  D:\masteruwefduyqeahfdqe\ASR-project\Pi

In [ ]:
files

In [260]:
len(set(files['file_id']).intersection(set(exp_features_pitts['file_id'])))

550

In [244]:
len(set(files['file_id']))

223

In [ ]:
# 1. Lexical / productivity
# unique_word_count
def normalize_word(w):
    w = str(w).lower()
    w = re.sub(r"[^A-Za-z0-9\u4e00-\u9fff']+", "", w)
    return w

def count_unique_words_from_series(text_series):
    words = []
    for text in text_series:
        for w in str(text).split():
            nw = normalize_word(w)
            if nw:
                words.append(nw)
    return len(set(words))

# word_count
# def compute_lexical_features(par_df, total_word_count):
#     unique_word_count = count_unique_words_from_series(par_df["text"])
#     print(f"Unique word count: {unique_word_count}, Total word count: {total_word_count}")
#     print(f"Text {par_df['text'].tolist()}")
#     type_token_ratio = unique_word_count / total_word_count if total_word_count > 0 else np.nan

#     return {
#         "unique_word_count": unique_word_count,
#         "type_token_ratio": type_token_ratio,
#     }
import re
import unicodedata


CHAT_NOISE_TOKENS = {
    "xxx", "yyy", "www", "unk", "unknown"
}

CJK_RE = re.compile(r"[\u4e00-\u9fff]")
GREEK_LATIN_RE = re.compile(
    r"[A-Za-zÀ-ÖØ-öø-ÿ\u0370-\u03FF\u1F00-\u1FFF]+"
)
CJK_OR_WORD_RE = re.compile(
    r"[\u4e00-\u9fff]|[A-Za-zÀ-ÖØ-öø-ÿ\u0370-\u03FF\u1F00-\u1FFF]+|\d+"
)


def clean_chat_text(text):
    if text is None:
        return ""

    # Handles np.nan safely
    if isinstance(text, float) and np.isnan(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKC", text)

    # Remove common CHAT annotations/noise
    text = re.sub(r"\[.*?\]", " ", text)      # removes [//], [/], [=! laugh], etc.
    text = re.sub(r"&=\S+", " ", text)        # removes &=laugh, &=cough, etc.
    text = re.sub(r"[<>/\\|_+=*~^]", " ", text)

    return text


def tokenize_text(text, language=None):
    text = clean_chat_text(text)
    language = str(language).lower() if language is not None else ""

    is_chinese = (
        language in ["mandarin", "chinese", "zh", "zh-cn", "zh-tw"]
        or CJK_RE.search(text) is not None
    )

    if is_chinese:
        # Better Chinese segmentation if jieba is installed.
        # If not installed, fallback = character-level Chinese tokens.
        try:
            import jieba
            tokens = [tok.strip() for tok in jieba.cut(text) if tok.strip()]
        except ImportError:
            tokens = CJK_OR_WORD_RE.findall(text)
    else:
        # Works for Greek and English alphabetic words
        tokens = GREEK_LATIN_RE.findall(text)

    tokens = [
        tok for tok in tokens
        if tok not in CHAT_NOISE_TOKENS and not tok.isdigit()
    ]

    return tokens


def count_tokens_from_series(text_series, language=None):
    all_tokens = []

    for text in text_series:
        all_tokens.extend(tokenize_text(text, language=language))

    return all_tokens

# def compute_lexical_features(par_df, language=None):
#     all_tokens = count_tokens_from_series(par_df["text"], language=language)

#     total_word_count = len(all_tokens)
#     unique_word_count = len(set(all_tokens))

#     type_token_ratio = (
#         unique_word_count / total_word_count
#         if total_word_count > 0
#         else np.nan
#     )

#     print(f"Language: {language}")
#     print(f"Total word count: {total_word_count}")
#     print(f"Unique word count: {unique_word_count}")
#     print(f"Type-token ratio: {type_token_ratio}")
#     print(f"First 50 tokens: {all_tokens[:50]}")

#     return {
#         "unique_word_count": unique_word_count,
#         "type_token_ratio": type_token_ratio,
#     }
def compute_lexical_features(par_df, language=None):
    all_tokens = []

    if "tokens" not in par_df.columns:
        # Fallback, just in case the function is called on an older dataframe
        for text in par_df["text"]:
            all_tokens.extend(tokenize_text(text, language=language))
    else:
        for toks in par_df["tokens"]:
            if isinstance(toks, list):
                all_tokens.extend(toks)

    total_word_count = len(all_tokens)
    unique_word_count = len(set(all_tokens))

    type_token_ratio = (
        unique_word_count / total_word_count
        if total_word_count > 0
        else np.nan
    )

    return {
        "unique_word_count": unique_word_count,
        "type_token_ratio": type_token_ratio,
    }
# word_rate_words_per_sec
def compute_utterance_word_rate_stats(par_df):
    utt_df = par_df.copy()

    # duration in seconds
    utt_df["duration_sec"] = utt_df["duration"] / 1000.0

    # valid utterances only
    utt_df = utt_df[
        utt_df["duration_sec"].notna() &
        (utt_df["duration_sec"] > 0)
    ].copy()

    utt_df["utterance_word_rate"] = utt_df["word_count"] / utt_df["duration_sec"]

    mean_utt_word_rate = utt_df["utterance_word_rate"].mean() if len(utt_df) > 0 else np.nan
    median_utt_word_rate = utt_df["utterance_word_rate"].median() if len(utt_df) > 0 else np.nan
    std_utt_word_rate = safe_std(utt_df["utterance_word_rate"])

    return {
        "mean_utterance_word_rate_words_per_sec": mean_utt_word_rate,
        "median_utterance_word_rate_words_per_sec": median_utt_word_rate,
        "std_utterance_word_rate_words_per_sec": std_utt_word_rate,
    }
# 2. Timing
# speech_duration_sec
# total recording duration 
# speaking_time_sec
# total participant speaking time from utterance spans
# total_speaking_time_normalized
def speaking_time(cha, par ):
    cha["duration"] = cha["time"].apply(
    lambda x: x[1] - x[0] if x is not None else None
    )
    total_speaking_time = cha[cha["speaker"].isin(par)]["duration"].sum()
    total_time = cha["time"].apply(lambda x: x[1] if x is not None else None).max()
    print(f"Total speaking time: {total_speaking_time:.2f} ms, Total time: {total_time:.2f} ms")
    total_speaking_time_normalized = total_speaking_time / total_time if total_time > 0 else 0
    print(f"Total speaking time normalized: {total_speaking_time_normalized:.4f}")
    return total_speaking_time, total_time, total_speaking_time_normalized

# speaking_time_sec / speech_duration_sec

# 4. Interaction
# interviewer_interruptions
def compute_interviewer_interruptions(cha_df, par_code, inv_code):
    """
    Counts interruptions as local overlap:
    previous utterance is PAR, current utterance is INV,
    and INV starts before PAR ends.
    """
    interruptions = 0
    cha_sorted = cha_df.sort_values("start_time", na_position="last").reset_index(drop=True)

    for i in range(1, len(cha_sorted)):
        prev_row = cha_sorted.iloc[i - 1]
        curr_row = cha_sorted.iloc[i]

        if (
            prev_row["speaker"] in par_code
            and curr_row["speaker"] == inv_code
            and pd.notna(prev_row["end_time"])
            and pd.notna(curr_row["start_time"])
            and curr_row["start_time"] < prev_row["end_time"]
        ):
            interruptions += 1

    return interruptions
# n_inv_utterances
# n_par_utterances
def compute_utterance_word_stats(par_df):
    n_par_utterances = len(par_df)

    mean_words_per_utterance = par_df["word_count"].mean() if n_par_utterances > 0 else np.nan
    median_words_per_utterance = par_df["word_count"].median() if n_par_utterances > 0 else np.nan
    std_words_per_utterance = par_df["word_count"].std() if n_par_utterances > 1 else 0.0
    mean_time_per_utterance = par_df["duration"].mean() if n_par_utterances > 0 else np.nan
    median_time_per_utterance = par_df["duration"].median() if n_par_utterances > 0 else np.nan
    std_time_per_utterance = par_df["duration"].std() if n_par_utterances > 1 else 0.0
    return {
        "n_par_utterances": n_par_utterances,
        "mean_words_per_utterance": mean_words_per_utterance,
        "median_words_per_utterance": median_words_per_utterance,
        "std_words_per_utterance": std_words_per_utterance,
        "mean_utterance_duration_sec": mean_time_per_utterance/1000.0 if not np.isnan(mean_time_per_utterance) else np.nan,
        "median_utterance_duration_sec": median_time_per_utterance/1000.0 if not np.isnan(median_time_per_utterance) else np.nan,
        "std_utterance_duration_sec": std_time_per_utterance/1000.0 if not np.isnan(std_time_per_utterance) else 0.0,
    }

def safe_mean(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return series.mean() if len(series) > 0 else np.nan
def safe_median(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return series.median() if len(series) > 0 else np.nan
def safe_std(series):
    series = pd.to_numeric(series, errors="coerce").dropna()

    if len(series) == 0:
        return np.nan

    # With one utterance, there is no variation.
    # Using 0.0 is usually better for ML than NaN.
    if len(series) == 1:
        return 0.0

    return series.std()


def compute_utterance_word_stats(par_df):
    n_par_utterances = len(par_df)

    duration_sec = pd.to_numeric(par_df["duration"], errors="coerce") / 1000.0

    return {
        "n_par_utterances": n_par_utterances,

        # Words per utterance
        "mean_words_per_utterance": safe_mean(par_df["word_count"]),
        "median_words_per_utterance": safe_median(par_df["word_count"]),
        "std_words_per_utterance": safe_std(par_df["word_count"]),

        # Unique words per utterance
        "mean_unique_words_per_utterance": safe_mean(par_df["unique_word_count_utt"]),
        "median_unique_words_per_utterance": safe_median(par_df["unique_word_count_utt"]),
        "std_unique_words_per_utterance": safe_std(par_df["unique_word_count_utt"]),

        # Utterance duration
        "mean_utterance_duration_sec": safe_mean(duration_sec),
        "median_utterance_duration_sec": safe_median(duration_sec),
        "std_utterance_duration_sec": safe_std(duration_sec),
    }

In [ ]:
# for each line collects features
# total duration speaking normalized by total duration of recording
# interviewer interruptions
# how many times they stopped
import pylangacq
feature_rows = []
for index, row in files.iterrows():
    print(row)
    wav_file = row['wav_path']
    cha_file = row["cha_path"]
    label = row["label"]
    print(wav_file, cha_file, label)
    # computing interpretable features from .cha files
    # 1. total duration of recording
    # 2. total duration of subject speaking
    try:
        reader = pylangacq.read_chat(cha_file,strict=False)
    except Exception as e:
        print(f"Could not read {cha_file}: {e}")
        continue
    
    rows_cha = []

    for utt in reader.utterances():
        text = " ".join(token.word for token in utt.tokens) if utt.tokens is not None else ""
        time_marks = utt.time_marks
        start_time = time_marks[0] if time_marks is not None else np.nan
        end_time = time_marks[1] if time_marks is not None else np.nan
        rows_cha.append({
            "speaker": utt.participant,
            "text": text,
            "time": utt.time_marks,
            "start_time": start_time,
            "end_time": end_time,

        })


    cha = pd.DataFrame(rows_cha)
    if cha.empty:
        print(f"Empty transcript: {cha_file}")
        continue
    print(cha)
    if language =="Mandarin":
        par = ["PAR0"]
        inv = "PAR1"
    elif language =="English":
        par = ["PAR"]
        inv = "INV"
    elif language =="Greek":
        par = ["PAR2","PAR1"]
        inv = "PAR0"
    cha["duration"] = cha["end_time"] - cha["start_time"]

    # cha["word_count"] = cha["text"].apply(lambda x: len(str(x).split()) if x is not None else 0)

    # Tokenize once and reuse everywhere.
    # This keeps word_count, lexical features, and utterance features consistent.
    cha["tokens"] = cha["text"].apply(lambda x: tokenize_text(x, language=language))
    cha["word_count"] = cha["tokens"].apply(len)
    cha["unique_word_count_utt"] = cha["tokens"].apply(lambda toks: len(set(toks)))

    # 1. total speaking duration and normalized speaking duration
    # getting total speaking miliseconds of par from the table
    total_speaking_time, total_time, total_speaking_time_normalized = speaking_time(cha, par)

    # 2. interviewer interruptions
    # count how many times the interviewer (INV) interrupts the subject (PAR)
    cha["is_par"] = cha["speaker"].isin(par)
    cha["is_inv"] = cha["speaker"] == inv

    par_df = cha[cha["is_par"]].copy()
    inv_df = cha[cha["is_inv"]].copy()

    word_count = par_df['word_count'].sum()
    # cha["par_end_time"] = cha.apply(lambda row: row["time"][1] if row["time"] is not None else None if row["is_par"] else None, axis=1)
    # cha["inv_start_time"] = cha.apply(lambda row: row["time"][0] if row["time"] is not None else None if row["is_inv"] else None, axis=1)
    # # this below might be wrong check
    # cha["inv_interrupts_par"] = cha.apply(lambda row: 1 if row["is_inv"] and any((cha["is_par"] & (cha["par_end_time"] > row["inv_start_time"])) ) else 0, axis=1)
    # interviewer_interruptions = cha["inv_interrupts_par"].sum()
    # print(f"Interviewer interruptions: {interviewer_interruptions}")
    interviewer_interruptions = 0
    cha_sorted = cha.sort_values("start_time", na_position="last").reset_index(drop=True)

    for i in range(1, len(cha_sorted)):
        prev_row = cha_sorted.iloc[i - 1]
        curr_row = cha_sorted.iloc[i]

        if (
            prev_row["speaker"] in par
            and curr_row["speaker"] == inv
            and pd.notna(prev_row["end_time"])
            and pd.notna(curr_row["start_time"])
            and curr_row["start_time"] < prev_row["end_time"]
        ):
            interviewer_interruptions += 1
    print(f"Interviewer interruptions: {interviewer_interruptions}")


    # 3. word rates
    word_rate_overall = (
    word_count / (total_speaking_time/ 1000)
    if pd.notna(total_speaking_time/ 1000) and total_speaking_time/ 1000 > 0
    else np.nan
    )

    utterance_word_rate_feats = compute_utterance_word_rate_stats(par_df)


    # word rate
    # count the number of words spoken by the subject
    # cha["word_count"] = cha["text"].apply(lambda x: len(x.split()) if x is not None else 0)
    # word_count = cha[cha["speaker"] == par]["word_count"].sum()
    # print(f"Word count: {par_df['word_count'].sum()}")
    # word_rate = par_df['word_count'].sum() / (total_speaking_time / 1000) if total_speaking_time > 0 else 0
    # print(f"Word rate: {word_rate:.2f} words/sec")


    # 4. unique word count and type-token ratio
    lexical_feats = compute_lexical_features(par_df, language=language)

    # 5. utterance structure
    utterance_word_stats = compute_utterance_word_stats(par_df)


    feature_row = {
        "sample_id": row["file_id"] if "file_id" in row else None,
        "wav_path": wav_file,
        "cha_path": cha_file,
        "label": label,
        "language": language,
        "total_time_ms": total_time,
        "total_speaking_time_ms": total_speaking_time,
        "total_speaking_time_normalized": total_speaking_time_normalized,
        "interviewer_interruptions": interviewer_interruptions,
        "word_count": word_count,
        "word_rate_words_per_sec": word_rate_overall,
    }

    feature_row.update(utterance_word_rate_feats)
    feature_row.update(lexical_feats)
    feature_row.update(utterance_word_stats)

    feature_rows.append(feature_row)

file_id                                                   001-0
wav_path      D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...
cha_path      D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...
label                                                         1
class_name                                             Dementia
Name: 0, dtype: object
D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\Dementia\wav_patient_only\001-0 (1)_patient.wav D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\Dementia\cha\001-0.cha 1
   speaker                                               text            time  \
0      INV                              this is the picture .    (1360, 2530)   
1      PAR                                              mhm .    (2821, 3211)   
2      INV  just tell me everything that you see happening...    (4022, 6646)   
3      PAR                                          alright .    (6650, 6820)   
4      PAR  there's a young boy that's getting a cookie jar .   (7277, 12026)   
5     

In [263]:
features_df=pd.DataFrame(feature_rows)

In [ ]:
features_df

In [ ]:
# change file id by adding _label
features_df['file_id'] = features_df.apply(lambda row: f"{row['file_id']}_{row['label']}" if pd.notna(row['file_id']) and pd.notna(row['label']) else row['file_id'], axis=1)
features_df

In [ ]:
exp_features_chinese['label'] = exp_features_chinese['wav_path'].apply(lambda x: 0 if "control" in x.lower() else 1)
exp_features_chinese

In [265]:
# change name of column sample_id to file_id for merging
features_df = features_df.rename(columns={"sample_id": "file_id"})
features_df

,file_id,wav_path,cha_path,label,language,total_time_ms,total_speaking_time_ms,total_speaking_time_normalized,interviewer_interruptions,word_count,...,n_par_utterances,mean_words_per_utterance,median_words_per_utterance,std_words_per_utterance,mean_unique_words_per_utterance,median_unique_words_per_utterance,std_unique_words_per_utterance,mean_utterance_duration_sec,median_utterance_duration_sec,std_utterance_duration_sec
0,001-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,54755.0,30708.0,0.560825,0,82,...,10,8.200000,9.0,4.638007,7.600000,8.0,4.247875,3.070800,3.4815,1.866022
1,001-2,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,80146.0,52713.0,0.657712,0,119,...,13,9.153846,8.0,4.896362,8.230769,8.0,4.265244,4.054846,4.4600,1.888797
2,003-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,235309.0,119001.0,0.505722,0,196,...,23,8.521739,7.0,6.148685,7.739130,6.0,5.065188,5.409136,4.0260,4.066340
3,005-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,82373.0,9792.0,0.118874,0,55,...,9,6.111111,6.0,2.934469,5.777778,6.0,2.438123,1.632000,1.6045,0.892242
4,005-2,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,54575.0,5548.0,0.101658,0,27,...,7,3.857143,5.0,2.734262,3.857143,5.0,2.734262,1.387000,1.3160,0.709348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546,686-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,0,English,72753.0,49775.0,0.684164,0,172,...,12,14.333333,10.5,11.452140,11.833333,9.5,7.720496,4.147917,3.5540,2.823484
547,688-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,0,English,50718.0,28859.0,0.569009,1,92,...,11,8.363636,7.0,5.143398,7.818182,7.0,4.600395,2.623545,2.7410,1.831573
548,691-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,0,English,69705.0,44735.0,0.641776,0,173,...,19,9.105263,8.0,4.920419,8.315789,8.0,4.083199,2.354474,1.8290,1.389643
549,709-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,0,English,32183.0,21589.0,0.670820,0,88,...,10,8.800000,6.0,6.494442,8.200000,6.0,5.493430,2.698625,2.6990,1.569371


In [ ]:
merged_features = pd.merge(features_df, exp_features_pitts.drop(columns=['language']), on=["file_id", "label"], how="left")
merged_features

In [ ]:
merged_features.keys()

In [267]:
merged_features.to_csv("full_feature_table_final_pitts.csv", index=False)

In [42]:
features_df.to_csv("interpretable_features_dummy_for_testing_bigger.csv", index=False)